# Data Cleaning

## Objective

The objective of this notebook is to clean the merged dataset by handling missing values, removing duplicate records, filtering invalid transactions, correcting data types, and preparing a clean dataset for exploratory data analysis and feature engineering.

In [60]:
import pandas as pd
import numpy as np

In [61]:
# Load merged dataset generated in the previous notebook
df = pd.read_csv("../data/interim/merged_data.csv")

# Display first five rows
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [62]:
# Create a copy for cleaning
df_copy = df.copy()

In [63]:
df_copy.shape

(1067371, 8)

In [64]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [65]:
df_copy.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

# Missing Values Analysis

The merged dataset contains **1,067,371 rows** and **8 columns**.

### Observations

- **Description** has **4,382 missing values**.
- **Customer ID** has **243,007 missing values**.
- All other columns have no missing values.

These missing values will be handled in the next steps based on their business importance.

In [66]:
# Count duplicate rows
duplicate_count = df_copy.duplicated().sum()

print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 34335


## Duplicate Records Analysis

The dataset contains **34,335 duplicate records**.

Duplicate records can negatively affect data quality and may lead to inaccurate analysis and model performance. Therefore, duplicate rows will be removed during the data cleaning process.

In [67]:
df_copy.dtypes

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

In [68]:
# Check records having Quantity <= 0
print("Quantity <= 0 :", (df_copy["Quantity"] <= 0).sum())

# Check records having Price <= 0
print("Price <= 0 :", (df_copy["Price"] <= 0).sum())

Quantity <= 0 : 22950
Price <= 0 : 6207


In [69]:
# Count cancelled invoices
cancelled_orders = df_copy["Invoice"].astype(str).str.startswith("C").sum()

print("Cancelled Orders:", cancelled_orders)

Cancelled Orders: 19494


In [70]:
# Dataset shape before removing duplicates
print("Shape Before:", df_copy.shape)

# Remove duplicate rows
df_copy.drop_duplicates(inplace=True)

# Dataset shape after removing duplicates
print("Shape After :", df_copy.shape)

# Verify duplicates
print("Remaining Duplicates:", df_copy.duplicated().sum())

Shape Before: (1067371, 8)
Shape After : (1033036, 8)
Remaining Duplicates: 0


In [71]:
# Convert InvoiceDate to datetime format
df_copy["InvoiceDate"] = pd.to_datetime(df_copy["InvoiceDate"])

In [72]:
df_copy.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

## Removing Cancelled Orders

Cancelled transactions are identified by invoice numbers starting with the letter **'C'**.

These records do not represent completed sales and may negatively impact sales analysis and predictive models. Therefore, they are removed from the dataset.

In [73]:
print("Before:", df_copy.shape)

df_copy = df_copy[~df_copy["Invoice"].astype(str).str.startswith("C")]

print("After :", df_copy.shape)

Before: (1033036, 8)
After : (1013932, 8)


## Removing Invalid Quantity

Transactions with zero or negative quantities do not represent valid product purchases.

Such records are removed to maintain data quality.

In [74]:
print("Before:", df_copy.shape)

df_copy = df_copy[df_copy["Quantity"] > 0]

print("After :", df_copy.shape)

Before: (1013932, 8)
After : (1010539, 8)


## Removing Invalid Price

Transactions with zero or negative prices are considered invalid for sales analysis.

These records are removed from the dataset.

In [75]:
print("Before:", df_copy.shape)

df_copy = df_copy[df_copy["Price"] > 0]

print("After :", df_copy.shape)

Before: (1010539, 8)
After : (1007913, 8)


## Handling Missing Values

The **Description** column contains only a small number of missing values, so these records are removed to maintain data quality.

The **Customer ID** column is essential for customer-level analysis, such as customer segmentation, churn prediction, and RFM analysis. Therefore, records with missing Customer IDs are also removed.

In [76]:
print("Before:", df_copy.shape)

df_copy.dropna(subset=["Description"], inplace=True)
df_copy.dropna(subset=["Customer ID"], inplace=True)

print("After :", df_copy.shape)

Before: (1007913, 8)
After : (779425, 8)


In [77]:
df_copy["Customer ID"] = df_copy["Customer ID"].astype(int)

In [78]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      779425 non-null  object        
 1   StockCode    779425 non-null  object        
 2   Description  779425 non-null  object        
 3   Quantity     779425 non-null  int64         
 4   InvoiceDate  779425 non-null  datetime64[ns]
 5   Price        779425 non-null  float64       
 6   Customer ID  779425 non-null  int64         
 7   Country      779425 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 53.5+ MB


In [81]:
# Check missing values
df_copy.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [82]:
# Check duplicate rows
df_copy.duplicated().sum()

np.int64(0)

In [83]:
# Check invalid Quantity
(df_copy["Quantity"] <= 0).sum()

np.int64(0)

In [84]:
# Check invalid Price
(df_copy["Price"] <= 0).sum()

np.int64(0)

In [85]:
print(df_copy.shape)

(779425, 8)


In [79]:
df_copy.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


## Final Dataset Summary

After cleaning, the dataset no longer contains duplicate records, invalid transactions, or missing values in critical columns. The cleaned dataset is ready for exploratory data analysis (EDA) and feature engineering.

In [80]:
df_copy.to_csv("../data/interim/cleaned_data.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


# Summary

### Completed Tasks

- Removed duplicate records.
- Converted `InvoiceDate` to datetime format.
- Removed cancelled transactions.
- Removed records with invalid quantity and price.
- Handled missing values in `Description` and `Customer ID`.
- Converted `Customer ID` to integer data type.
- Saved the cleaned dataset for further analysis.

### Output

`data/interim/cleaned_data.csv`

### Next Step

The cleaned dataset will be used in **03_exploratory_data_analysis.ipynb**.